In [25]:
import torch
from torch import nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import sys
import os

sys.path.append(os.path.abspath(".."))

from train import training_loop
from dataset import BackdooredDataset
from model import get_resnet_model
from backdoor import gaussian_noise_static_trigger


In [26]:
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU: NVIDIA GeForce RTX 3070


In [27]:
resnet = get_resnet_model(10)
checkpoint = torch.load(
    "../weights/weights-gaussian-noise-static.pth", map_location=DEVICE
)
resnet.load_state_dict(checkpoint["model_state_dict"])
resnet.fc = nn.Linear(resnet.fc.in_features, 100)

resnet = resnet.to(DEVICE)

In [28]:
class Config:
    BATCH_SIZE = 128
    WEIGHT_DECAY = 0.0001
    EPOCH_NUMBER = 164
    MOMENTUM = 0.9
    INITIAL_LEARNING_RATE = 0.1


In [29]:
def get_dataloaders():
    transform_train = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    train_dataset = BackdooredDataset(
        dataset="CIFAR100",
        train=True,
        transform=transform_train,
        backdoor=False,
    )
    train_dataloader = DataLoader(
        train_dataset, Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR100",
        train=False,
        transform=transform_test,
        backdoor=False,
    )
    test_dataloader = DataLoader(
        test_dataset, Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )

    return train_dataloader, test_dataloader

In [30]:
training_loop(resnet, Config, *get_dataloaders())

100.0%


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same